# Gold archetype cluster assignments

In [4]:
import sys
from pathlib import Path

import pandas as pd
import s3fs
from collections import defaultdict
import re
from datetime import date, timedelta

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.ml.archetype_clustering import (
    numeric_feature_columns,
    prepare_dataframe_for_archetype_clustering,
)

pd.set_option("display.max_columns", None)

## Batter

In [5]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"  # default GOLD_PREFIX in src/pipeline/settings.py

role = "batter"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_archetypes.parquet"
with fs.open(path, "rb") as f:
    df_batter_archetypes = pd.read_parquet(f)

pd.set_option("display.max_columns", None)
df_batter_archetypes.head()

,player_id,year,role,n_pitches_total,swing_rate,zone_swing_rate,chase_rate,contact_rate,whiff_rate,barrel_rate,hard_hit_rate,pull_percent,opposite_field_percent,gb_percent,ld_percent,fb_percent,iffb_percent,sweet_spot_percent,launch_speed_mean,launch_speed_iqr,launch_angle_mean,launch_angle_iqr,iso_value_mean,estimated_slg_using_speedangle_mean,estimated_woba_using_speedangle_mean,sprint_speed_mean,def_oaa_total,def_adj_estimated_fielding_success_rate_mean,def_outfield_catch_completion_rate,def_arm_strength_max_mph,def_pop_time_2b_sec,def_framing_runs,def_drs_total,def_framing_runs_was_missing,def_pop_time_2b_sec_was_missing,cluster_id
0,456781,2025,batter,774,0.512920,0.326873,0.186047,0.391473,0.121447,0.010830,0.223827,0.379310,0.282759,0.482759,0.248276,0.200000,0.068966,0.306859,83.162816,19.400,17.787004,42.00,0.095000,0.434493,0.258718,25.0,1.0,0.77,0.0,72.1,0.0,0.0,0.0,1,1,2
1,457705,2025,batter,2344,0.453925,0.349829,0.104096,0.342577,0.111348,0.016598,0.257261,0.419355,0.182796,0.428954,0.219839,0.270777,0.080429,0.295172,83.360719,21.850,16.840000,45.00,0.106830,0.537488,0.333453,27.1,0.0,0.00,0.0,0.0,0.0,0.0,0.0,1,1,2
2,457759,2025,batter,822,0.479319,0.336983,0.142336,0.403893,0.075426,0.006623,0.168874,0.321678,0.258741,0.384615,0.244755,0.286713,0.083916,0.294702,82.212252,18.075,24.238411,38.75,0.080808,0.467290,0.298232,25.3,0.0,0.00,0.0,75.7,0.0,0.0,0.0,1,1,2
3,467793,2025,batter,2078,0.445621,0.321944,0.123677,0.345043,0.100577,0.007669,0.263804,0.502907,0.151163,0.409884,0.220930,0.261628,0.107558,0.279141,83.118558,23.200,18.113497,52.25,0.095808,0.447878,0.290137,25.6,8.0,0.75,0.0,87.1,0.0,0.0,0.0,1,1,5
4,500743,2025,batter,1267,0.451460,0.315706,0.135754,0.376480,0.074980,0.015766,0.213964,0.368030,0.226766,0.457249,0.241636,0.234201,0.066914,0.295964,81.318694,20.650,14.941704,45.00,0.122093,0.434925,0.296853,25.9,7.0,0.78,0.0,86.8,0.0,0.0,0.0,1,1,5


### Batter cluster profiles

Mean values of `numeric_feature_columns()` (same exclusions as PCA / GMM) by `cluster_id`. Use this table to pick short labels for each cluster. Edit `batter_cluster_names` below to record them.

In [6]:
_df_b = prepare_dataframe_for_archetype_clustering(df_batter_archetypes)
_feat_b = [c for c in numeric_feature_columns(_df_b) if c != "cluster_id"]
_feat_b = [c for c in _feat_b if c in _df_b.columns]

_gb_b = _df_b.groupby("cluster_id", sort=True)
batter_n_players = _gb_b.size().rename("n_players")
batter_feature_means_by_cluster = _gb_b[_feat_b].mean()

batter_cluster_names: dict[int, str] = {
    # e.g. 0: "high-whiff chase profile",
}

print("Players per cluster")
display(batter_n_players.to_frame())
print("Mean features by cluster (rows = features, columns = cluster_id)")
display(batter_feature_means_by_cluster.T)
if batter_cluster_names:
    print("Same table with your labels as column names")
    display(
        batter_feature_means_by_cluster.rename(columns=batter_cluster_names).T
    )

Players per cluster


,n_players
cluster_id,
0,41
1,51
2,72
3,87
4,23
5,118


Mean features by cluster (rows = features, columns = cluster_id)


cluster_id,0,1,2,3,4,5
barrel_rate,0.020830,0.015611,0.015852,0.015302,0.009834,0.015559
chase_rate,0.148957,0.147521,0.139248,0.138161,0.164683,0.151326
contact_rate,0.365877,0.371111,0.364696,0.371328,0.398801,0.367488
def_adj_estimated_fielding_success_rate_mean,0.735488,0.000000,0.226528,0.775920,0.636087,0.586441
def_arm_strength_max_mph,91.441463,1.380392,73.277778,88.724138,91.456522,90.521186
def_drs_total,0.536585,0.254902,-0.111111,-0.678161,5.478261,0.211864
def_framing_runs,0.000000,0.040044,0.000000,0.000000,0.000000,0.000000
def_oaa_total,1.024390,0.000000,0.069444,-0.609195,8.000000,-0.296610
def_outfield_catch_completion_rate,0.317934,0.000000,0.046322,0.134807,0.290574,0.159036
def_pop_time_2b_sec,0.000000,1.956471,0.028750,0.000000,0.000000,0.000000


## Pitcher

In [7]:
BUCKET = "diamond-dna"
prefix = "gold/statcast"

role = "pitcher"
year = "2025"

fs = s3fs.S3FileSystem()
path = f"s3://{BUCKET}/{prefix}/{role}/year={year}/player_year_archetypes.parquet"
with fs.open(path, "rb") as f:
    df_pitcher_archetypes = pd.read_parquet(f)

df_pitcher_archetypes.head()

,player_id,year,role,n_pitches_total,batter_swing_rate,batter_zone_swing_rate,batter_chase_rate,batter_contact_rate,batter_whiff_rate,in_zone_rate,release_speed_max,fastball_velo_mean,offspeed_velo_mean,velo_differential,release_speed_iqr,release_spin_rate_iqr,pfx_x_iqr,release_extension_mean,release_extension_iqr,pfx_z_mean,pfx_z_iqr,plate_x_mean,plate_x_sd,plate_z_mean,plate_z_sd,edge_percent,first_pitch_strike_rate,xwoba_allowed_lhb_mean,xwoba_allowed_rhb_mean,platoon_xwoba_allowed_diff,gb_percent_allowed,ld_percent_allowed,fb_percent_allowed,iffb_percent_allowed,delta_run_exp_mean,pt_CH_release_speed_mean,pt_CH_pfx_x_mean,pt_FC_release_speed_mean,pt_FC_pfx_x_mean,pt_CU_release_speed_mean,pt_CU_pfx_x_mean,pt_SI_release_speed_mean,pt_SI_pfx_x_mean,pt_FF_release_speed_mean,pt_FF_pfx_x_mean,pt_NONE_release_speed_mean,pt_NONE_release_spin_rate_mean,pt_NONE_pfx_x_mean,pt_CS_pfx_x_mean,pitch_type_CU_share,pitch_type_SI_share,pitch_type_FC_share,pitch_type_FF_share,pitch_type_CH_share,pitch_type_CS_share,pitch_type_SL_share,pitch_type_entropy,pt_SL_release_speed_mean,pt_SL_pfx_x_mean,pt_ST_release_speed_mean,pt_ST_pfx_x_mean,pitch_type_ST_share,pitch_type_PO_share,pt_KC_release_speed_mean,pt_KC_pfx_x_mean,pt_FS_release_speed_mean,pt_FS_pfx_x_mean,pitch_type_FS_share,pitch_type_KC_share,pitch_type_SV_share,pt_SV_release_speed_mean,pt_SV_pfx_x_mean,pitch_type_FA_share,pitch_type_EP_share,pitch_type_UN_share,pitch_type_KN_share,pt_CH_pfx_x_mean_was_missing,pt_CH_release_speed_mean_was_missing,pt_CH_release_spin_rate_mean_was_missing,pt_CS_pfx_x_mean_was_missing,pt_CS_release_speed_mean_was_missing,pt_CS_release_spin_rate_mean_was_missing,pt_CU_pfx_x_mean_was_missing,pt_CU_release_speed_mean_was_missing,pt_CU_release_spin_rate_mean_was_missing,pt_FC_pfx_x_mean_was_missing,pt_FC_release_speed_mean_was_missing,pt_FC_release_spin_rate_mean_was_missing,pt_FF_pfx_x_mean_was_missing,pt_FF_release_speed_mean_was_missing,pt_FF_release_spin_rate_mean_was_missing,pt_FO_pfx_x_mean_was_missing,pt_FO_release_speed_mean_was_missing,pt_FO_release_spin_rate_mean_was_missing,pt_FS_pfx_x_mean_was_missing,pt_FS_release_speed_mean_was_missing,pt_FS_release_spin_rate_mean_was_missing,pt_KC_pfx_x_mean_was_missing,pt_KC_release_speed_mean_was_missing,pt_KC_release_spin_rate_mean_was_missing,pt_KN_pfx_x_mean_was_missing,pt_KN_release_speed_mean_was_missing,pt_KN_release_spin_rate_mean_was_missing,pt_NONE_pfx_x_mean_was_missing,pt_NONE_release_speed_mean_was_missing,pt_NONE_release_spin_rate_mean_was_missing,pt_SC_pfx_x_mean_was_missing,pt_SC_release_speed_mean_was_missing,pt_SC_release_spin_rate_mean_was_missing,pt_SI_pfx_x_mean_was_missing,pt_SI_release_speed_mean_was_missing,pt_SI_release_spin_rate_mean_was_missing,pt_SL_pfx_x_mean_was_missing,pt_SL_release_speed_mean_was_missing,pt_SL_release_spin_rate_mean_was_missing,pt_ST_pfx_x_mean_was_missing,pt_ST_release_speed_mean_was_missing,pt_ST_release_spin_rate_mean_was_missing,pt_SV_pfx_x_mean_was_missing,pt_SV_release_speed_mean_was_missing,pt_SV_release_spin_rate_mean_was_missing,cluster_id
0,434378,2025,pitcher,2775,0.483243,0.336577,0.146667,0.376937,0.106306,0.491892,98.3,93.926526,83.441730,10.484796,9.9,190.75,1.2700,6.016940,0.3,0.721243,1.300,0.113814,0.774437,2.469420,0.992253,0.833700,0.623762,0.373694,0.361138,0.012556,0.357430,0.251004,0.303213,0.088353,-0.000224,84.695614,-1.079561,0.000000,0.000000,78.457403,0.625481,0.000000,0.000000,93.931911,-0.739491,0.0,0.0,0.00,0.0,0.143336,0.003723,0.000000,0.453835,0.084885,0.0,0.231943,1.411599,87.108186,0.339021,80.495475,0.965656,0.082278,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,0,1,1,1,2
1,445276,2025,pitcher,925,0.550270,0.427027,0.123243,0.432432,0.117838,0.563243,96.9,92.750964,83.039535,9.711429,2.6,153.50,0.3100,6.880131,0.2,1.405131,0.190,-0.123323,0.655352,2.673284,0.961436,0.815739,0.627530,0.390560,0.337013,0.053547,0.303030,0.16

### Pitcher cluster profiles

Mean values of `numeric_feature_columns()` by `cluster_id` for pitchers. Edit `pitcher_cluster_names` to record labels.

In [8]:
_df_p = prepare_dataframe_for_archetype_clustering(df_pitcher_archetypes)
_feat_p = [c for c in numeric_feature_columns(_df_p) if c != "cluster_id"]
_feat_p = [c for c in _feat_p if c in _df_p.columns]

_gb_p = _df_p.groupby("cluster_id", sort=True)
pitcher_n_players = _gb_p.size().rename("n_players")
pitcher_feature_means_by_cluster = _gb_p[_feat_p].mean()

pitcher_cluster_names: dict[int, str] = {
    # e.g. 0: "low-velo command",
}

print("Players per cluster")
display(pitcher_n_players.to_frame())
print("Mean features by cluster (rows = features, columns = cluster_id)")
display(pitcher_feature_means_by_cluster.T)
if pitcher_cluster_names:
    print("Same table with your labels as column names")
    display(
        pitcher_feature_means_by_cluster.rename(columns=pitcher_cluster_names).T
    )

Players per cluster


,n_players
cluster_id,
0,71
1,80
2,144
3,45
4,63
5,91


Mean features by cluster (rows = features, columns = cluster_id)


cluster_id,0,1,2,3,4,5
batter_chase_rate,0.140897,0.159328,0.135936,0.137499,0.146155,0.150995
batter_contact_rate,0.367664,0.338802,0.374200,0.336530,0.370670,0.375487
batter_swing_rate,0.461051,0.466713,0.479778,0.439482,0.479001,0.507738
batter_whiff_rate,0.093387,0.127910,0.105579,0.102952,0.108332,0.132251
batter_zone_swing_rate,0.320154,0.307385,0.343842,0.301983,0.332846,0.356743
delta_run_exp_mean,0.005710,-0.002173,0.000852,-0.000469,-0.000925,-0.005354
edge_percent,0.850382,0.852776,0.842671,0.843132,0.838184,0.846036
fastball_velo_mean,91.763071,94.860725,93.578472,93.849230,94.372991,94.786257
fb_percent_allowed,0.274887,0.255428,0.275359,0.239424,0.236425,0.289125
first_pitch_strike_rate,0.619403,0.592142,0.619249,0.591742,0.629927,0.639749
